# Raman Kyn / Trp: intensity ratio vs. concentration ratio

Pipeline
1. Import raw Raman `.txt` files (one file per group) - you are prompted for paths.
2. Enter `(C_kyn, C_trp)` for each file -> concentration ratio `[kyn/trp]`.
3. Intensity / 157.5, then baseline (background) removal -> new column.
4. Kyn peak = max in 550-570 cm-1, Trp peak = max in 745-755 cm-1 -> `I560/I750`.
5. Match concentration ratio <-> intensity ratio.
6. Five models (PLSR+MLP, PLSR+SVR, PLSR+KRR, PCA+MLP, PCA+SVR) -> power law `I560/I750 = a * ([kyn]/[trp])^b`.
7. Enter a new `(C_kyn, C_trp)` -> predicted intensity ratio.

Run the cells top to bottom. Cells 3 and 7 are interactive (`input()` prompts).

In [ ]:
import re, glob, os, ast, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import sparse
from scipy.sparse.linalg import spsolve
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cross_decomposition import PLSRegression
from sklearn.neural_network import MLPRegressor
from sklearn.svm import SVR
from sklearn.kernel_ridge import KernelRidge
from sklearn.compose import TransformedTargetRegressor
from sklearn.model_selection import GridSearchCV, KFold
from sklearn.metrics import r2_score
warnings.filterwarnings("ignore")

# ---- settings you may tweak ----
DIVISOR     = 157.5          # intensity normalisation factor
KYN_WINDOW  = (550, 570)     # cm-1
TRP_WINDOW  = (745, 755)     # cm-1
ALS_LAMBDA  = 1e5            # baseline smoothness (larger = stiffer baseline)
ALS_P       = 0.01           # baseline asymmetry (smaller = hugs the bottom more)
ALS_ITER    = 10

## 1. Helper functions

In [ ]:
_num = re.compile(r"[-+]?(?:\d+\.?\d*|\.\d+)(?:[eE][-+]?\d+)?")

def load_raman_txt(path):
    """Read a 2+ column text file (wavenumber, intensity). Header/non-numeric lines are skipped."""
    xs, ys = [], []
    with open(path, "r", errors="ignore") as f:
        for line in f:
            nums = _num.findall(line)
            toks = re.split(r"[\s,;]+", line.strip())
            if len(nums) >= 2 and len(toks) >= 2 and all(_num.fullmatch(t) for t in toks[:2]):
                xs.append(float(toks[0])); ys.append(float(toks[1]))
    if len(xs) < 10:
        raise ValueError(f"Could not read numeric (wavenumber, intensity) columns from {path}")
    df = pd.DataFrame({"wavenumber": xs, "intensity": ys}).sort_values("wavenumber").reset_index(drop=True)
    return df

def als_baseline(y, lam=ALS_LAMBDA, p=ALS_P, niter=ALS_ITER):
    """Asymmetric least-squares baseline (Eilers & Boelens)."""
    L = len(y)
    D = sparse.diags([1, -2, 1], [0, -1, -2], shape=(L, L - 2))
    DDT = lam * D.dot(D.transpose())
    w = np.ones(L)
    for _ in range(niter):
        W = sparse.spdiags(w, 0, L, L)
        z = spsolve((W + DDT).tocsc(), w * y)
        w = p * (y > z) + (1 - p) * (y < z)
    return z

def process_spectrum(df):
    df = df.copy()
    df["scaled"] = df["intensity"] / DIVISOR
    df["baseline"] = als_baseline(df["scaled"].values)
    df["edited"] = df["scaled"] - df["baseline"]
    return df

def find_peak(df, window, col="edited"):
    m = (df["wavenumber"] >= window[0]) & (df["wavenumber"] <= window[1])
    if not m.any():
        raise ValueError(f"No data points in window {window}")
    sub = df[m]
    i = sub[col].idxmax()
    return df.loc[i, "wavenumber"], df.loc[i, col]

def parse_tuple(s):
    t = ast.literal_eval(s.strip())
    if not (isinstance(t, (tuple, list)) and len(t) == 2):
        raise ValueError("need two numbers")
    ck, ct = float(t[0]), float(t[1])
    if ck <= 0 or ct <= 0:
        raise ValueError("concentrations must be > 0")
    return ck, ct

def ask_tuple(prompt):
    while True:
        try:
            return parse_tuple(input(prompt))
        except Exception as e:
            print(f"  Invalid input ({e}). Enter like (0.5, 1.0)")

## 2. Import files and enter concentrations
Type a file path (a glob such as `data/*.txt` also works). Press Enter on an empty line when you have added all files.

In [ ]:
files = []
while True:
    s = input("Path to raw Raman .txt file (blank = done): ").strip().strip("\"'")
    if not s:
        if files: break
        print("  Please add at least one file."); continue
    hits = sorted(glob.glob(os.path.expanduser(s)))
    if not hits:
        print("  File not found."); continue
    files.extend(h for h in hits if h not in files)
    print(f"  {len(files)} file(s) so far")

groups = []
for f in files:
    ckyn, ctrp = ask_tuple(f"(C_kyn, C_trp) for {os.path.basename(f)}: ")
    groups.append(dict(file=os.path.basename(f), path=f, C_kyn=ckyn, C_trp=ctrp, conc_ratio=ckyn / ctrp))
pd.DataFrame(groups)[["file", "C_kyn", "C_trp", "conc_ratio"]]

## 3. Normalise, remove background, find peaks, intensity ratio

In [ ]:
spectra = {}
for g in groups:
    df = process_spectrum(load_raman_txt(g["path"]))
    spectra[g["file"]] = df
    g["kyn_pos"], g["I_kyn"] = find_peak(df, KYN_WINDOW)
    g["trp_pos"], g["I_trp"] = find_peak(df, TRP_WINDOW)
    g["I_ratio"] = g["I_kyn"] / g["I_trp"]

data = pd.DataFrame(groups).drop(columns="path")
display(data) if "display" in globals() else print(data)

fig, axes = plt.subplots(1, len(groups), figsize=(4.5 * len(groups), 3.5), squeeze=False)
for ax, g in zip(axes[0], groups):
    d = spectra[g["file"]]
    ax.plot(d.wavenumber, d.scaled, lw=.8, label="scaled")
    ax.plot(d.wavenumber, d.baseline, lw=.8, label="baseline")
    ax.plot(d.wavenumber, d.edited, lw=1, label="edited")
    ax.plot([g["kyn_pos"], g["trp_pos"]], [g["I_kyn"], g["I_trp"]], "rv")
    ax.set_title(g["file"], fontsize=9); ax.set_xlabel("Raman shift (cm$^{-1}$)")
axes[0][0].legend(fontsize=7); plt.tight_layout(); plt.show()

## 4. Machine-learning models and power law

Model inputs are the log-concentrations and log-ratio `[log C_kyn, log C_trp, log(C_kyn/C_trp)]`; the target is `log(I560/I750)`.
Each model is `StandardScaler -> PLSR or PCA (dimension reduction) -> MLP / SVR / Kernel ridge`, with hyper-parameters picked by cross-validated grid search.
The power law `I = a * R^b` is then obtained by regressing `log I` on `log R` using each model's fitted values over the measured concentration range
(a smooth dense grid along the measured range, using the median log C_trp). A direct fit to the raw data is shown as a reference.

> With only a handful of groups the models are very data-limited; add more concentration groups for trustworthy results.

In [ ]:
class PLSReducer(BaseEstimator, TransformerMixin):
    """PLSRegression used as a supervised dimension-reduction step (returns X scores)."""
    def __init__(self, n_components=2): self.n_components = n_components
    def fit(self, X, y=None):
        k = min(self.n_components, X.shape[1], X.shape[0] - 1)
        self.pls_ = PLSRegression(n_components=max(k, 1), scale=False).fit(X, y)
        return self
    def transform(self, X): return self.pls_.transform(X)

def features(ckyn, ctrp):
    ckyn, ctrp = np.asarray(ckyn, float), np.asarray(ctrp, float)
    return np.column_stack([np.log10(ckyn), np.log10(ctrp), np.log10(ckyn / ctrp)])

X = features(data.C_kyn, data.C_trp)
y = np.log10(data.I_ratio.values)
n = len(y)
if n < 4:
    print(f"WARNING: only {n} groups - cross-validation / fits will be unreliable.")
cv = KFold(n_splits=min(5, n), shuffle=True, random_state=0)

def make(reducer, regressor, grid):
    red = PLSReducer() if reducer == "PLSR" else PCA()
    reg, g = {
        "MLP": (MLPRegressor(max_iter=5000, random_state=0, solver="lbfgs"),
                {"regressor__hidden_layer_sizes": [(5,), (10,), (10, 5)], "regressor__alpha": [1e-4, 1e-2, 1e-1]}),
        "SVR": (SVR(kernel="rbf"),
                {"regressor__C": [0.1, 1, 10, 100], "regressor__epsilon": [0.001, 0.01, 0.1], "regressor__gamma": ["scale", 0.1, 1]}),
        "KRR": (KernelRidge(kernel="rbf"),
                {"regressor__alpha": [1e-3, 1e-2, 1e-1, 1], "regressor__gamma": [0.01, 0.1, 1]}),
    }[regressor]
    pipe = Pipeline([("scale", StandardScaler()), ("reduce", red), ("regressor", reg)])
    ttr = TransformedTargetRegressor(regressor=pipe, transformer=StandardScaler())
    grid_all = {"regressor__reduce__n_components": [k for k in (1, 2, 3) if k <= X.shape[1]]}
    grid_all.update({"regressor__" + k: v for k, v in g.items()})
    return GridSearchCV(ttr, grid_all, cv=cv, scoring="neg_mean_squared_error", n_jobs=1)

MODELS = {
    "PLSR + MLP": ("PLSR", "MLP"), "PLSR + SVR": ("PLSR", "SVR"), "PLSR + KRR": ("PLSR", "KRR"),
    "PCA + MLP": ("PCA", "MLP"),   "PCA + SVR": ("PCA", "SVR"),
}
fitted = {name: make(*spec, None).fit(X, y) for name, spec in MODELS.items()}

def power_law(R, I):
    b, loga = np.polyfit(np.log10(R), np.log10(I), 1)
    pred = loga + b * np.log10(R)
    return 10 ** loga, b, r2_score(np.log10(I), pred)

R_meas = data.conc_ratio.values
lc_trp = np.median(np.log10(data.C_trp))
R_grid = np.logspace(np.log10(R_meas.min()), np.log10(R_meas.max()), 100)
Xg = features(R_grid * 10 ** lc_trp, np.full_like(R_grid, 10 ** lc_trp))

rows, curves = [], {}
a, b, r2 = power_law(R_meas, data.I_ratio.values)
rows.append(dict(model="Direct data fit", a=a, b=b, R2_powerlaw=r2, CV_RMSE_log10=np.nan, best_params=""))
for name, gs in fitted.items():
    Ig = 10 ** gs.predict(Xg); curves[name] = Ig
    a, b, r2 = power_law(R_grid, Ig)
    rows.append(dict(model=name, a=a, b=b, R2_powerlaw=r2, CV_RMSE_log10=np.sqrt(-gs.best_score_),
                     best_params={k.replace("regressor__", ""): v for k, v in gs.best_params_.items()}))
results = pd.DataFrame(rows).set_index("model")
pd.set_option("display.max_colwidth", 120)
display(results) if "display" in globals() else print(results)
for m, r in results.iterrows():
    print(f"{m:16s}: I560/I750 = {r.a:.4g} * ([kyn]/[trp])^{r.b:.4g}")

plt.figure(figsize=(6, 4.5))
plt.scatter(R_meas, data.I_ratio, c="k", zorder=5, label="measured")
for name, Ig in curves.items():
    a, b = results.loc[name, ["a", "b"]]; plt.plot(R_grid, a * R_grid ** b, label=name)
plt.xscale("log"); plt.yscale("log"); plt.xlabel("[kyn]/[trp]"); plt.ylabel("I560/I750"); plt.legend(fontsize=8); plt.tight_layout(); plt.show()

## 5. Predict the intensity ratio for a new (C_kyn, C_trp)
Re-run this cell for as many new combinations as you like. Leave blank to stop.

In [ ]:
while True:
    s = input("New (C_kyn, C_trp) (blank = stop): ").strip()
    if not s: break
    try:
        ck, ct = parse_tuple(s)
    except Exception as e:
        print(f"  Invalid input ({e}). Enter like (0.5, 1.0)"); continue
    R = ck / ct
    xn = features([ck], [ct])
    out = {name: 10 ** gs.predict(xn)[0] for name, gs in fitted.items()}
    pl = {f"{name} (power law)": results.loc[name, "a"] * R ** results.loc[name, "b"] for name in fitted}
    print(f"  concentration ratio [kyn/trp] = {R:.4g}"
          + ("   (outside the measured range - extrapolation!)" if not R_meas.min() <= R <= R_meas.max() else ""))
    print(pd.DataFrame({"predicted I560/I750 (model)": out, "predicted I560/I750 (power law)": pl.values()}, index=list(out)).round(4).to_string())